# Chicago Crime Arrest Prediction
## Luca Accomando
## Spring 2026
## Data Mining — Classification Project

## Project Workflow
This notebook follows an industry-style analytics workflow:

1. **Problem Framing & Data Acquisition**
2. **Exploratory Data Analysis (EDA) & Data Preparation**
3. **Model Development, Evaluation & Business Interpretation**

## GitHub + Colab Workflow
1. Create a **new GitHub repository** for your project.
2. Upload this notebook to your repository.
3. In GitHub, open the notebook in **Google Colab**.
4. Commit changes to GitHub as you work.
5. Submit your GitHub repository link when requested.

## Project Requirements
- Use a **classification dataset** ✅
- Use **Random Forest** as one of your main models ✅
- Use **Google Colab** ✅
- Include **visualization, preparation, modeling, and interpretation** ✅
- Explain results in a way a manager or stakeholder could understand ✅

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AutoViz
!pip install autoviz -q
from autoviz.AutoViz_Class import AutoViz_Class

# scikit-learn tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, classification_report, cohen_kappa_score)

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

---
# Deliverable 1: Problem Framing & Data Acquisition

## What problem are you trying to solve?

Each year, the Chicago Police Department records hundreds of thousands of crime incidents across the city. However, **not all reported crimes result in an arrest**. This project builds a binary classification model to predict whether a reported crime will result in an arrest (True/False), using features such as crime type, location, time of day, and district.

Understanding what drives arrest outcomes allows city officials, police leadership, and researchers to surface data-driven patterns in policing — and to evaluate where resources or policy adjustments may be needed.

## What is the target variable?

- **Column:** `arrest`
- **Type:** Boolean → Binary classification
- **Values:** `True` (arrest made) = 1, `False` (no arrest) = 0

## What organization, industry, or domain could use this model?

| Stakeholder | Use Case |
|---|---|
| Chicago Police Department | Identify crime types and districts with low arrest rates for resource review |
| City Policy Analysts | Study patterns in arrest outcomes across neighborhoods |
| Criminal Justice Researchers | Benchmark fairness across geography and crime category |
| Community Organizations | Understand gaps in public safety outcomes |

## Why does this problem matter?

Arrest rate is a key performance indicator in public safety. If certain crime types or locations consistently show low arrest rates, that may indicate resource gaps, patrol inefficiencies, or structural disparities. A predictive model helps quantify those patterns and supports better decision-making.

## Where did the dataset come from?

- **Source:** [Chicago Data Portal — Crimes 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)
- **Maintained by:** City of Chicago (updated regularly)
- **Size:** 8+ million rows; sampled for modeling efficiency
- **Features:** Crime type, location description, date/time, district, community area, latitude/longitude, domestic flag, and arrest outcome

## Why did you choose this dataset?

The Chicago Crime Dataset was selected because it has the richest ecosystem of any dataset reviewed:
- Widely used in Kaggle competitions, Medium tutorials, and academic projects (e.g., Notre Dame)
- Hosted on an official government open data portal — reliable and regularly updated
- Contains temporal, geospatial, and categorical features ideal for feature engineering
- The target variable (`arrest`) is clean, binary, and has direct real-world meaning
- Large enough (8M+ rows) to support robust modeling while sampling keeps compute manageable

## Data Loading



In [ ]:
# Option A: Load directly from Chicago Data Portal (sample via Socrata API)
data_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=100000"
df = pd.read_csv(data_url)

# Option B: Load from your GitHub repo after uploading the CSV
# data_url = "https://raw.githubusercontent.com/yourusername/chicago-crime-arrest-prediction/main/data/crimes_sample.csv"
# df = pd.read_csv(data_url)

df.info()

In [ ]:
df.head()

---
# Deliverable 2: Exploratory Data Analysis (EDA) & Data Preparation

## What to include
- Basic shape and structure of the data
- Variable types
- Missing values
- Class balance of the target
- Visualizations that help explain the data
- Preparation steps used before modeling

## Questions to resolve
- Are there missing values?
- Are the classes balanced?
- Which variables might be useful predictors?
- Are any variables likely to cause problems?
- Do I need to eliminate any variables due to correlation, redundancy, or uniqueness (e.g., ID columns)?

In [ ]:
# Basic data inspection
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)

In [ ]:
# Missing values summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
display(missing_summary)

In [ ]:
# Target variable: arrest
target = "arrest"

# Class balance
display(df[target].value_counts(dropna=False))
display(df[target].value_counts(normalize=True, dropna=False))

## AutoViz — Automated EDA

AutoViz generates many plots at once for fast exploratory analysis.

**Important for Colab:** After AutoViz runs, use `plt.close('all')` before creating your own plots to prevent old figures from reappearing.

In [ ]:
from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()

In [ ]:
# Run AutoViz on a sample to avoid memory issues
df_av = df.sample(10000, random_state=42)

AV.AutoViz(
    filename='',
    sep=',',
    depVar=target,
    dfte=df_av,
    header=0,
    verbose=1,
    lowess=False,
    chart_format='svg',
    max_rows_analyzed=10000,
    max_cols_analyzed=20
)

plt.close('all')

## Custom EDA Visualizations

In [ ]:
# Arrest rate by Primary Crime Type
arrest_by_type = (
    df.groupby('primary_type')[target]
    .apply(lambda x: (x == True).mean() * 100)
    .sort_values(ascending=False)
    .reset_index()
)
arrest_by_type.columns = ['primary_type', 'arrest_rate']

plt.figure(figsize=(14, 7))
sns.barplot(data=arrest_by_type, x='arrest_rate', y='primary_type', palette='Blues_d')
plt.title('Arrest Rate by Crime Type (%)')
plt.xlabel('Arrest Rate (%)')
plt.ylabel('Crime Type')
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering: extract time components from date
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Arrest rate by hour of day
arrest_by_hour = df.groupby('hour')[target].apply(lambda x: (x == True).mean() * 100)

plt.figure(figsize=(12, 5))
arrest_by_hour.plot(kind='line', marker='o', color='steelblue')
plt.title('Arrest Rate by Hour of Day')
plt.xlabel('Hour (0 = Midnight, 12 = Noon)')
plt.ylabel('Arrest Rate (%)')
plt.tight_layout()
plt.show()

## Data Preparation

In [ ]:
# Select features
features = ['primary_type', 'location_description', 'domestic', 'district', 'hour', 'day_of_week', 'month']

df_model = df[features + [target]].dropna().copy()

# Encode target as integer
df_model[target] = df_model[target].astype(int)

X = df_model[features]
y = df_model[target]

print(f'Features: {features}')
print(f'Dataset size: {X.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True).round(3)}')

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

print(f'Training size: {X_train.shape[0]}')
print(f'Test size: {X_test.shape[0]}')

In [ ]:
# Preprocessing pipeline
categorical_features = ['primary_type', 'location_description']
numeric_features = ['district', 'hour', 'day_of_week', 'month']
boolean_features = ['domestic']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features),
    ('bool', SimpleImputer(strategy='most_frequent'), boolean_features)
])

---
# Deliverable 3: Model Development, Evaluation & Interpretation

We compare multiple classification models, with Random Forest as the primary model. All models use the same preprocessing pipeline for a fair comparison.

In [ ]:
# Compare multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=123),
    'Decision Tree': DecisionTreeClassifier(random_state=123),
    'Random Forest': RandomForestClassifier(random_state=123),
    'Gradient Boosting': GradientBoostingClassifier(random_state=123),
    'KNN': KNeighborsClassifier()
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'F1 (weighted)': round(f1_score(y_test, preds, average='weighted'), 4),
        'Kappa': round(cohen_kappa_score(y_test, preds), 4)
    })

results_df = pd.DataFrame(results).sort_values('F1 (weighted)', ascending=False)
display(results_df)

## Random Forest — Primary Model

In [ ]:
# Train Random Forest
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=123))
])

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred))
print("Cohen's Kappa:", round(cohen_kappa_score(y_test, y_pred), 4))

In [ ]:
# Confusion matrix
plt.close('all')
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Arrest', 'Arrest'])
disp.plot(ax=ax)
ax.set_title('Random Forest Confusion Matrix')
plt.show()

## Hyperparameter Tuning

We don't know the best settings ahead of time, so we try multiple combinations. GridSearchCV tests combinations and selects the best version based on cross-validation performance.

In [ ]:
# Tune Random Forest
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

rf_tuning_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=123))
])

grid_search = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

print('Best Parameters:', grid_search.best_params_)

In [ ]:
# Final evaluation on test set
best_preds = best_rf.predict(X_test)

print('Tuned Random Forest Classification Report:')
print(classification_report(y_test, best_preds))

kappa = cohen_kappa_score(y_test, best_preds)
print("Cohen's Kappa:", round(kappa, 4))

In [ ]:
# Tuned confusion matrix
plt.close('all')
cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Arrest', 'Arrest'])
disp.plot(ax=ax)
ax.set_title('Tuned Random Forest Confusion Matrix')
plt.show()

## Feature Importance

Feature importance shows which inputs influenced the Random Forest most.

**Important caveats:**
- Importance does **not** prove causation
- Importance can be split across multiple one-hot encoded columns
- Importance tells us what mattered to the model, not necessarily what matters in the real world

In [ ]:
# Feature importance from tuned model
feature_names = best_rf.named_steps['preprocessor'].get_feature_names_out()
importances = best_rf.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

display(feature_importance_df.head(15))

In [ ]:
# Plot top feature importances
top_n = 15
top_features = feature_importance_df.head(top_n).sort_values('importance')

plt.figure(figsize=(8, 6))
plt.barh(top_features['feature'], top_features['importance'], color='steelblue')
plt.title(f'Top {top_n} Feature Importances — Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Interpretation Prompts

Write your answers below in plain language — as if explaining to a manager or city official:
- How well did the model perform overall?
- Which class was easier or harder to predict — arrests or non-arrests?
- Which variables seemed most important to the model?
- Where did the model make mistakes? What might explain those errors?
- How could this model actually be used by the Chicago Police Department or a city analyst?
- What would you improve or do differently next time?

### Student Interpretation Summary

*Replace this section with your written interpretation after running the model.*

---
## Optional: Save Model and Cleaned Data

In [ ]:
import joblib

# Save trained model
# joblib.dump(best_rf, 'final_model.pkl')
# print('Model saved.')

# Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df_model.to_csv('/content/drive/MyDrive/chicago_crimes_cleaned.csv', index=False)